# Embedding-Only Commodity Grouping within HS10

**Experiment question:** can the *existing* Qdrant dense embeddings of goods descriptions
(`dense__dense`, 1024-dim, L2-normalized representations of `ACCURATE_NAME`) identify
meaningful commodity groups **within each HS10 code**, and do those groups materially
improve `PRICE_KG` homogeneity compared with the current `HS10 + origin + transport`
comparison populations?

## Ground rules of this experiment

1. **Embedding-only.** No new embeddings are generated, no embedding model is loaded,
   and no TF-IDF / SVD / n-gram / lexical features are used. We deliberately test
   whether the existing Qdrant vectors alone are sufficient.
2. **Price never enters clustering.** The pipeline is strictly
   `description embedding → cluster membership`, and only afterwards
   `cluster membership → analyze PRICE_KG`. Price, country, and transport are
   **evaluation variables only** — they are never clustering features. We do sweep the
   number of clusters `k` and use price homogeneity to *evaluate* each cut, but the
   membership of every row at a given `k` is determined by the embedding alone.
3. **Clustering happens separately inside each HS10.** Never across HS10 codes.
4. **Multi-SKU weighting.** Rows sharing `DECL_ID | CMDT_ID | G32` are one line-level
   price observation; every price-based statistic uses `weight = 1 / n_rows_in_line`.
   Rows are **not** deduplicated — clustering membership stays row-level.
5. **Experiment, not production.** No undervaluation flags, underpricing rules, smart
   segments, LL/UL, or targeting logic — only the grouping question above.

> **Provenance note:** this repository contains no earlier notebook, so the weighting
> and evaluation formulas referenced by the experiment brief (line-level weights,
> `log(PRICE_KG)`, robust spread `IQR/1.349`, adjusted eta-squared, coverage) are
> implemented here directly from that brief, which defines them fully.

**Workflow:** cells run one at a time. Cells 11–29 are a deep walkthrough of a single
HS10 (`WALKTHROUGH_HS` — change it and re-run those cells for any other code); the
batch section then runs the identical pipeline over every selected HS10.

In [ ]:
# Imports, library versions (reproducibility), and plot styling
import json
import pickle
import time
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, fcluster
import sklearn
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib
import matplotlib.pyplot as plt
from cycler import cycler
from IPython.display import display

for _m in (np, pd, scipy, sklearn, pa, matplotlib):
    print(f"{_m.__name__:<12} {_m.__version__}")

# Validated categorical palette + neutral chart chrome (light surface).
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
           "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS_C, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
STATUS_COLORS = {          # status roles, always shown WITH a text label, never color alone
    "STRONG_SUCCESS": "#0ca30c",
    "USEFUL": "#2a78d6",
    "NEUTRAL": "#898781",
    "INSUFFICIENT_DESCRIPTION": "#ec835a",
    "OVERFRAGMENTED": "#d03b3b",
}
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "figure.autolayout": True,
    "axes.edgecolor": AXIS_C, "axes.linewidth": 1.0,
    "axes.labelcolor": INK2, "axes.titlecolor": INK, "text.color": INK,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.prop_cycle": cycler(color=PALETTE),
    "font.size": 10, "axes.titlesize": 11, "figure.titlesize": 13,
    "legend.frameon": False, "lines.linewidth": 1.8,
})
print("Plot style configured.")

In [ ]:
# ============================== CONFIGURATION ==============================
# Change values here and re-run the cells that depend on them.

DATA_DIR = "data"          # directory containing the Qdrant parquet exports
CACHE_DIR = "cache"        # cached hierarchies / k-sweeps / group assignments
OUTPUT_DIR = "outputs"     # final exports

MODE = "selected_hs"       # "selected_hs" (load only SELECTED_HS_CODES) or "batch"
SELECTED_HS_CODES = [
    # expected successes
    "6109100000", "6115950000", "6111209000",
    # neutral / control
    "2005202000",
    # difficult
    "4016930005", "3304990000", "3926909709",
    "7326909807", "8708299001", "8708999701",
]
WALKTHROUGH_HS = "6109100000"   # HS10 used for the single-code deep walkthrough

# ---- k-sweep and selection --------------------------------------------------
K_MIN = 2
K_MAX = 60
MIN_GROUP_CNT = 20         # minimum group size for a group to be "usable"
MIN_COVERAGE = 0.60        # min weighted share of observations in usable groups
ETA_TOL = 0.02             # pick the SMALLEST k within this tolerance of the best adj. eta^2
TARGET_SPREAD = 0.35       # target robust spread of log(PRICE_KG), natural-log units
MIXED_SPREAD_FACTOR = 1.5  # group flagged MIXED if spread > factor * TARGET_SPREAD
MIN_SPREAD_N = 5           # min price-valid rows needed to compute a group spread

# ---- scalability ------------------------------------------------------------
LARGE_HS_THRESHOLD = 15000 # above this row count use the micro-cluster APPROXIMATION
N_MICRO_MAX = 1500         # max MiniBatchKMeans micro-clusters on the large path
SILHOUETTE_SAMPLE = 5000   # sample size for the (diagnostic-only) silhouette score
RUN_APPROX_COMPARISON = True   # compare exact vs micro-approx on a sample when the
APPROX_COMPARE_SAMPLE = 8000   # micro path is used; sample size for that comparison
APPROX_ARI_GOOD = 0.70     # adjusted-Rand threshold for "approximation agrees closely"

# ---- batch mode -------------------------------------------------------------
BATCH_MIN_N = 500          # batch mode: only HS10 codes with at least this many rows
MAX_LOAD_ROWS = 400_000    # batch mode: cap on total rows loaded in one run

# ---- HS10 classification thresholds ----------------------------------------
STRONG_REDUCTION = 0.40    # dispersion reduction for STRONG_SUCCESS
USEFUL_REDUCTION = 0.20    # dispersion reduction for USEFUL
INSUFFICIENT_SPREAD = 0.60 # reference: grouped spread above this is clearly unresolved
MAX_SMALL_GROUP_SHARE = 0.50   # OVERFRAGMENTED if more than this share of groups is tiny

# ---- display ----------------------------------------------------------------
N_REPRESENTATIVES = 7      # representative descriptions per group
TOP_GROUPS_DISPLAY = 10    # groups shown in representative-description tables
BOX_TOP_GROUPS = 12        # groups shown in the grouped price-distribution chart

RANDOM_STATE = 42
USE_CACHE = True

print(f"MODE={MODE} | {len(SELECTED_HS_CODES)} selected HS codes | "
      f"k in [{K_MIN}, {K_MAX}] | target spread {TARGET_SPREAD}")

In [ ]:
# Locate the parquet files; fail loudly and helpfully if the data is missing.
DATA_PATH = Path(DATA_DIR)
CACHE_PATH = Path(CACHE_DIR)
OUTPUT_PATH = Path(OUTPUT_DIR)
CACHE_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

parquet_files = sorted(DATA_PATH.glob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(
        "\n".join([
            "",
            f"No parquet files found in DATA_DIR = '{DATA_PATH.resolve()}'.",
            "",
            "This notebook expects the Qdrant parquet exports (columns: point_id,",
            "payload_json, dense__dense) to be available locally.",
            "Either copy/symlink the export files into that directory, or edit",
            "DATA_DIR in the configuration cell to point at their real location,",
            "then re-run this cell.",
        ])
    )

_rows = []
for _f in parquet_files:
    _pf = pq.ParquetFile(_f)
    _rows.append({"file": _f.name,
                  "rows": _pf.metadata.num_rows,
                  "size_MB": round(_f.stat().st_size / 1e6, 1)})
files_df = pd.DataFrame(_rows)
display(files_df)
print(f"{len(parquet_files)} parquet files, {files_df['rows'].sum():,} total rows, "
      f"{files_df['size_MB'].sum():,.0f} MB on disk")

## Loading strategy

`HS_CODE` lives *inside* the `payload_json` string, so parquet predicate push-down
cannot filter on it. Loading all ~1.4M rows x 1024 floats just to keep a few HS codes
would waste several GB of memory. Instead the loader streams every file in small
record batches and, per batch:

1. runs one **vectorized regex** over `payload_json` to extract the 10-digit
   `HS_CODE` (no JSON parsing for non-matching rows — this is the cheap pre-filter);
2. fully `json.loads`-parses **only the matching rows** and keeps only the metadata
   columns the experiment needs;
3. materializes the `dense__dense` batch (fixed-size-list fast path, plain-list
   fallback) and keeps only the matching rows as `float32`.

Peak transient memory is one batch (~32 MB of embeddings); retained memory is only
the selected HS codes. `MODE="batch"` first does a payload-only counting pass, then
loads the eligible codes largest-first up to `MAX_LOAD_ROWS`.

In [ ]:
# Loader functions
HS_RE = r'"HS_CODE"\s*:\s*"?(\d{10})"?'
LOAD_COLUMNS = ["point_id", "payload_json", "dense__dense"]
META_KEYS = ["DECL_ID", "CMDT_ID", "G32", "DT_ID", "DT_NO", "HS_CODE", "PRICE_KG",
             "ACCURATE_NAME", "MANUFACTURE_COUNTRY_CODE", "TRANSPORT"]
EXPECTED_DIM = 1024


def embedding_matrix(col):
    """pyarrow list column -> (n, dim) float32 matrix. No re-embedding, no math —
    just a memory-layout conversion of the existing vectors."""
    n = len(col)
    if col.null_count:
        raise ValueError(f"dense__dense contains {col.null_count} null vectors")
    flat = col.flatten().to_numpy(zero_copy_only=False)
    if n == 0 or flat.size % n:
        raise ValueError(f"ragged dense__dense column ({flat.size} values / {n} rows)")
    return flat.reshape(n, flat.size // n).astype(np.float32, copy=False)


def line_key(df):
    """Multi-SKU line key: rows sharing DECL_ID|CMDT_ID|G32 are ONE price observation."""
    return (df["DECL_ID"].fillna("").astype(str) + "|"
            + df["CMDT_ID"].fillna("").astype(str) + "|"
            + df["G32"].fillna("").astype(str))


def normalize_hs(s):
    return (s.astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(10))


def scan_hs_counts(files, batch_size=16384):
    """Light pass: payload_json only, regex HS extraction, row counts per HS10."""
    counts = {}
    for f in files:
        for batch in pq.ParquetFile(f).iter_batches(batch_size=batch_size,
                                                    columns=["payload_json"]):
            hs = batch.column("payload_json").to_pandas().str.extract(HS_RE, expand=False)
            for code_, cnt in hs.value_counts().items():
                counts[code_] = counts.get(code_, 0) + int(cnt)
    return pd.Series(counts, dtype="int64").sort_values(ascending=False)


def load_hs_rows(files, wanted, batch_size=8192):
    """Stream all files, keep only rows whose HS_CODE is in `wanted`."""
    wanted = {str(h) for h in wanted}
    metas, embs = [], []
    scanned = 0
    t0 = time.time()
    for f in files:
        for batch in pq.ParquetFile(f).iter_batches(batch_size=batch_size,
                                                    columns=LOAD_COLUMNS):
            scanned += len(batch)
            payload = batch.column("payload_json").to_pandas()
            hs = payload.str.extract(HS_RE, expand=False)
            mask = hs.isin(wanted).to_numpy()
            if not mask.any():
                continue
            idx = np.flatnonzero(mask)
            pids = batch.column("point_id").take(pa.array(idx)).to_pylist()
            recs = []
            for j, i in enumerate(idx):
                d = json.loads(payload.iloc[int(i)])
                rec = {k: d.get(k) for k in META_KEYS}
                rec["point_id"] = str(pids[j])
                recs.append(rec)
            metas.append(pd.DataFrame(recs))
            embs.append(embedding_matrix(batch.column("dense__dense"))[idx])
    if not metas:
        return (pd.DataFrame(columns=META_KEYS + ["point_id"]),
                np.empty((0, EXPECTED_DIM), np.float32), scanned)
    meta = pd.concat(metas, ignore_index=True)
    meta["HS_CODE"] = normalize_hs(meta["HS_CODE"])
    X = np.vstack(embs)
    print(f"Scanned {scanned:,} rows in {time.time() - t0:.1f}s -> kept {len(meta):,}")
    return meta, X, scanned


def split_by_hs(meta, X):
    out = {}
    for hs, g in meta.groupby("HS_CODE", sort=False):
        out[str(hs)] = {"meta": g.reset_index(drop=True), "X": X[g.index.to_numpy()]}
    return out

print("Loader functions defined.")

In [ ]:
# Load the data for the chosen MODE and show a per-HS10 summary.
if MODE == "selected_hs":
    wanted_codes = [str(h) for h in SELECTED_HS_CODES]
elif MODE == "batch":
    print("Batch mode: counting rows per HS10 (payload-only pass)...")
    hs_counts = scan_hs_counts(parquet_files)
    eligible = hs_counts[hs_counts >= BATCH_MIN_N]
    wanted_codes, _running = [], 0
    for code_, cnt in eligible.items():
        if _running + cnt > MAX_LOAD_ROWS:
            continue
        wanted_codes.append(code_)
        _running += int(cnt)
    skipped = len(eligible) - len(wanted_codes)
    print(f"{len(eligible)} HS codes have >= {BATCH_MIN_N} rows; loading "
          f"{len(wanted_codes)} of them (~{_running:,} rows, MAX_LOAD_ROWS="
          f"{MAX_LOAD_ROWS:,}); {skipped} skipped this run — raise MAX_LOAD_ROWS "
          f"or run again with a different selection to cover them.")
else:
    raise ValueError(f"MODE must be 'selected_hs' or 'batch', got {MODE!r}")

meta_all, X_all, _scanned = load_hs_rows(parquet_files, wanted_codes)
if len(meta_all) == 0:
    raise RuntimeError(
        "\n".join([
            "",
            f"None of the requested HS codes were found in '{DATA_PATH.resolve()}'.",
            f"Requested: {wanted_codes}",
            "Check that DATA_DIR points at the right export and that the codes are",
            "10-digit strings as they appear in payload_json.",
        ])
    )

data = split_by_hs(meta_all, X_all)
missing_codes = [c for c in wanted_codes if c not in data]
if missing_codes:
    print(f"WARNING: no rows found for {len(missing_codes)} requested HS codes: "
          f"{missing_codes}")
del meta_all, X_all  # per-HS slices are independent copies

_rows = []
for hs, d in data.items():
    m = d["meta"]
    price = pd.to_numeric(m["PRICE_KG"], errors="coerce").to_numpy(dtype=float)
    _rows.append({
        "HS_CODE": hs,
        "n_rows": len(m),
        "n_line_obs": int(line_key(m).nunique()),
        "price_valid_share": round(float(np.mean(np.isfinite(price) & (price > 0))), 4),
        "emb_MB": round(d["X"].nbytes / 1e6, 1),
    })
load_summary = pd.DataFrame(_rows).sort_values("n_rows", ascending=False,
                                               ignore_index=True)
display(load_summary)
print(f"Loaded {load_summary['n_rows'].sum():,} rows across {len(data)} HS10 codes "
      f"({load_summary['emb_MB'].sum():,.0f} MB of embeddings)")

In [ ]:
# Data-quality checks: embedding shape/norms, NaNs, duplicates, price validity.
# The vectors are expected to be 1024-dim and L2-normalized; we VERIFY, we do not
# regenerate. Only if norms deviate materially are copies re-scaled (a safety net
# that changes nothing for correctly exported data).
_checks = []
for hs, d in data.items():
    X = d["X"]
    norms = np.linalg.norm(X, axis=1)
    off = np.abs(norms - 1.0) > 1e-3
    _checks.append({
        "HS_CODE": hs,
        "dim": X.shape[1],
        "n_nan_vectors": int(np.isnan(X).any(axis=1).sum()),
        "norm_min": round(float(norms.min()), 5),
        "norm_max": round(float(norms.max()), 5),
        "n_norm_off": int(off.sum()),
        "dup_point_id": int(d["meta"]["point_id"].duplicated().sum()),
        "missing_name": int(d["meta"]["ACCURATE_NAME"].isna().sum()),
    })
    if off.any():
        good = norms > 0
        X[good] = X[good] / norms[good, None]
        print(f"NOTE: HS10 {hs}: re-scaled {int(off.sum())} vectors whose L2 norm "
              f"deviated from 1 by >1e-3 (cosine geometry requires unit norms; "
              f"the vectors themselves are unchanged in direction).")
quality_df = pd.DataFrame(_checks)
display(quality_df)

_bad_dim = quality_df[quality_df["dim"] != EXPECTED_DIM]
if len(_bad_dim):
    print(f"WARNING: unexpected embedding dimension in {list(_bad_dim['HS_CODE'])} "
          f"(expected {EXPECTED_DIM}). Proceeding with the actual dimension.")
if quality_df["n_nan_vectors"].sum() == 0 and quality_df["n_norm_off"].sum() == 0:
    print("All embeddings are finite and L2-normalized — used as-is, no modification.")

## Methodology: metrics and weighting

**Price variable:** `y = ln(PRICE_KG)`, computed only for *price-valid* rows
(`PRICE_KG` finite and > 0). Invalid-price rows are still clustered and exported —
they are excluded only from price statistics.

**Multi-SKU line weight:** `w = 1 / (# rows sharing DECL_ID|CMDT_ID|G32)`. Every
price statistic below is weighted with `w`, so one declaration line listing four SKUs
contributes one observation of price influence, not four. `W = Σw` therefore equals
the number of line-level observations.

### Category A — embedding/clustering metrics (price-free)

- number of groups, group-size distribution (min / median / P10), share of tiny
  groups (`< MIN_GROUP_CNT`);
- **silhouette score** (cosine distance, fixed random sample) — **diagnostic only**:
  it describes embedding geometry and is *never* an input to k-selection, HS10
  classification, or group membership.

### Category B — economic evaluation metrics (price, strictly post-clustering)

- per-group **robust spread** `= (Q75_w − Q25_w)(ln PRICE_KG) / 1.349`
  (weighted IQR of log price; ≈ σ for a normal distribution);
- **grouped spread** = weight-weighted mean of group spreads over *qualifying* groups
  (raw size ≥ `MIN_GROUP_CNT`, spread computable), plus the weighted median and P90
  of group spreads;
- **weighted eta²** `= 1 − SS_within/SS_total` on `y` (weighted), and
  **adjusted eta²** `= 1 − (1 − eta²)(W − 1)/(W − k − 1)` which penalizes the group
  count the way adjusted R² penalizes regressors;
- **coverage** = weighted share of observations sitting in groups of raw size
  ≥ `MIN_GROUP_CNT`;
- share of observations in groups with spread ≤ `TARGET_SPREAD`;
- **baseline** = the whole HS10 as one group;
  `dispersion_reduction = 1 − grouped_spread / baseline_spread`.

Weighted quantiles use an interpolated midpoint rule (`cum(w) − w/2`), implemented
locally so results do not depend on the installed numpy version.

In [ ]:
# Metric helper functions (+ self-tests at the bottom of the cell)

def weighted_quantile(values, q, weights):
    """Interpolated weighted quantiles (midpoint rule). Returns np.array like q."""
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    q = np.atleast_1d(np.asarray(q, dtype=float))
    ok = np.isfinite(v) & np.isfinite(w) & (w > 0)
    v, w = v[ok], w[ok]
    if v.size == 0:
        return np.full(q.shape, np.nan)
    order = np.argsort(v)
    v, w = v[order], w[order]
    cw = (np.cumsum(w) - 0.5 * w) / np.sum(w)
    return np.interp(q, cw, v)


def robust_spread(y, w, min_n=None):
    """Weighted IQR/1.349 of y (= ln PRICE_KG). NaN when too few valid rows."""
    min_n = MIN_SPREAD_N if min_n is None else min_n
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    ok = np.isfinite(y) & np.isfinite(w) & (w > 0)
    if ok.sum() < min_n:
        return np.nan
    q25, q75 = weighted_quantile(y[ok], [0.25, 0.75], w[ok])
    return float((q75 - q25) / 1.349)


def weighted_eta_squared(y, w, labels):
    """Weighted eta^2 of y across `labels`. Returns (eta2, W, k_data)."""
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    labels = np.asarray(labels)
    if y.size == 0 or w.sum() <= 0:
        return np.nan, 0.0, 0
    W = float(w.sum())
    ybar = float(np.sum(w * y) / W)
    ss_tot = float(np.sum(w * (y - ybar) ** 2))
    if ss_tot <= 0:
        return np.nan, W, 0
    agg = pd.DataFrame({"g": labels, "wy": w * y, "w": w}).groupby("g").sum()
    gmean = (agg["wy"] / agg["w"])
    ghat = pd.Series(labels).map(gmean).to_numpy(dtype=float)
    ss_win = float(np.sum(w * (y - ghat) ** 2))
    return 1.0 - ss_win / ss_tot, W, int(len(agg))


def adjusted_eta_squared(eta2, W, k_data):
    """Adjusted-R^2-style penalty on the group count; W = effective sample size."""
    if not np.isfinite(eta2) or W - k_data - 1 <= 0:
        return np.nan
    return float(1.0 - (1.0 - eta2) * (W - 1.0) / (W - k_data - 1.0))


def prepare_frame(meta):
    """Add PRICE_KG_NUM, PRICE_VALID, LOG_PRICE and the multi-SKU OBS_WEIGHT."""
    df = meta.copy()
    price = pd.to_numeric(df["PRICE_KG"], errors="coerce").to_numpy(dtype=float)
    valid = np.isfinite(price) & (price > 0)
    df["PRICE_KG_NUM"] = price
    df["PRICE_VALID"] = valid
    lp = np.full(len(df), np.nan)
    lp[valid] = np.log(price[valid])
    df["LOG_PRICE"] = lp
    key = line_key(df)
    df["OBS_WEIGHT"] = (1.0 / key.groupby(key).transform("size")).to_numpy(dtype=float)
    return df


def baseline_stats(frame):
    """The whole HS10 as ONE group — the current comparison population."""
    v = frame["PRICE_VALID"].to_numpy()
    return {
        "n_rows": int(len(frame)),
        "n_lines": float(frame["OBS_WEIGHT"].sum()),
        "price_valid_share": float(v.mean()) if len(frame) else np.nan,
        "baseline_spread": robust_spread(frame["LOG_PRICE"].to_numpy()[v],
                                         frame["OBS_WEIGHT"].to_numpy()[v]),
        "baseline_coverage": 1.0 if len(frame) >= MIN_GROUP_CNT else 0.0,
    }

# ---- self-tests -------------------------------------------------------------
_v = np.arange(1.0, 101.0)
assert abs(weighted_quantile(_v, [0.5], np.ones(100))[0] - 50.5) < 1e-9
_q = weighted_quantile(np.array([1.0, 2.0, 3.0]), [0.5], np.array([1.0, 1.0, 10.0]))[0]
assert _q > 2.5, "heavy weight on 3.0 must pull the median up"
_e, _W, _k = weighted_eta_squared(
    np.array([0.0, 0.1, 5.0, 5.1]), np.ones(4), np.array([1, 1, 2, 2]))
assert _e > 0.99 and _k == 2, "two well-separated groups must give eta^2 ~ 1"
_toy = pd.DataFrame({"DECL_ID": ["a", "a", "b"], "CMDT_ID": [1, 1, 1],
                     "G32": [1, 1, 1], "PRICE_KG": [2.0, 2.0, 3.0]})
_tw = prepare_frame(_toy)["OBS_WEIGHT"].to_numpy()
assert np.allclose(_tw, [0.5, 0.5, 1.0]), "multi-SKU rows must share one unit of weight"
assert np.isnan(robust_spread(np.array([1.0, 2.0]), np.array([1.0, 1.0]), min_n=5))
print("Metric helpers defined — all self-tests passed.")

## Single-HS10 walkthrough

The next cells dissect one HS10 (`WALKTHROUGH_HS`) step by step: baseline price
dispersion, hierarchy construction, k-sweep, k-selection, final groups,
representative descriptions, and classification. To inspect a different code, change
`WALKTHROUGH_HS` in the configuration cell and re-run from the next cell onward —
the batch section further down runs this identical pipeline for every loaded HS10.

In [ ]:
# Prepare the walkthrough frame and baseline (pre-grouping) price statistics.
if WALKTHROUGH_HS in data:
    wt_hs = WALKTHROUGH_HS
else:
    wt_hs = max(data, key=lambda h: len(data[h]["meta"]))
    print(f"WARNING: WALKTHROUGH_HS={WALKTHROUGH_HS!r} is not in the loaded data — "
          f"falling back to the largest loaded code {wt_hs!r}. Fix the config to "
          f"analyze the intended HS10.")

wt_meta = data[wt_hs]["meta"]
wt_X = data[wt_hs]["X"]
wt = prepare_frame(wt_meta)
wt_base = baseline_stats(wt)

print(f"HS10 {wt_hs} — walkthrough")
print(f"  embeddings                : {wt_X.shape[0]:,} x {wt_X.shape[1]} "
      f"(float32, L2 norms in [{np.linalg.norm(wt_X, axis=1).min():.4f}, "
      f"{np.linalg.norm(wt_X, axis=1).max():.4f}])")
print(f"  rows                      : {wt_base['n_rows']:,}")
print(f"  line-level observations W : {wt_base['n_lines']:,.1f}")
print(f"  price-valid share         : {wt_base['price_valid_share']:.1%}")
print(f"  BASELINE robust spread    : {wt_base['baseline_spread']:.3f}  "
      f"(target {TARGET_SPREAD})")
print(f"  BASELINE usable coverage  : {wt_base['baseline_coverage']:.1%}")

In [ ]:
# Baseline price distribution: the current comparison population (whole HS10).
def plot_baseline_hist(frame, hs, base, ax=None):
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(7.5, 3.4))
    v = frame["PRICE_VALID"].to_numpy()
    y = frame["LOG_PRICE"].to_numpy()[v]
    w = frame["OBS_WEIGHT"].to_numpy()[v]
    med = weighted_quantile(y, [0.5], w)[0]
    ax.hist(y, bins=60, weights=w, color=PALETTE[0], edgecolor=SURFACE,
            linewidth=0.4)
    ax.axvspan(med - TARGET_SPREAD, med + TARGET_SPREAD, color=GRID, alpha=0.55,
               zorder=0, label=f"median ± target spread ({TARGET_SPREAD})")
    ax.axvline(med, color=INK2, linewidth=1.2, linestyle="--")
    ax.set_xlabel("ln(PRICE_KG)")
    ax.set_ylabel("line-weighted count")
    if standalone:
        ax.set_title(f"HS10 {hs} — baseline price distribution "
                     f"(robust spread {base['baseline_spread']:.3f})")
    else:
        ax.set_title(f"baseline prices (spread {base['baseline_spread']:.2f})")
    ax.legend(loc="upper right", fontsize=8)
    if standalone:
        plt.show()

plot_baseline_hist(wt, wt_hs, wt_base)

## Clustering method

**Reference method (exact):** hierarchical agglomerative clustering with **cosine
distance** and **average linkage**. We build the full dendrogram once with
`scipy.cluster.hierarchy.linkage(pdist(X, "cosine"), method="average")` and cut it at
every candidate `k` with `fcluster(..., criterion="maxclust")`. This is exactly
equivalent to fitting `sklearn.AgglomerativeClustering(metric="cosine",
linkage="average", n_clusters=k)` separately for each `k`, but one O(n²) build serves
the entire k-sweep. Cosine is the natural metric for these L2-normalized vectors, and
average linkage is valid for arbitrary dissimilarities (Ward would require Euclidean).
Memory is the binding constraint: the condensed distance matrix at n = 15,000 is
~0.9 GB — on machines with little RAM, lower `LARGE_HS_THRESHOLD`.

**Large HS10 codes (> `LARGE_HS_THRESHOLD` rows): a documented APPROXIMATION.**
O(n²) is not feasible, so we use MiniBatchKMeans micro-clustering
(`n_micro = min(N_MICRO_MAX, max(50, n/10))`; Euclidean k-means on unit vectors is
equivalent to spherical k-means since ‖u−v‖² = 2−2·cosθ), then run the exact
average-linkage hierarchy over the **micro-cluster centroids** (re-normalized to unit
length — cosine is norm-invariant so this changes no distances, it only keeps later
dot-product code valid), and map every row to a final group through its micro-cluster.
The linkage is *not* size-weighted: weighting would let dense generic-description
regions dominate the merge order and swallow small but distinct products.

This micro-cluster path is **not equivalent to the exact method** — rows can only
separate along micro-cluster boundaries. Its results are therefore labeled
`micro-approx` everywhere (vs `exact`), and when it is used, an explicit
**approximation-quality check** re-runs both methods on a manageable sample and
reports the adjusted Rand index and side-by-side price metrics, so the approximation
is never silently presented as the reference result.

In [ ]:
# Clustering, caching, and diagnostics functions.

def build_hierarchy_small(X):
    """EXACT reference method: average-linkage hierarchy on cosine distances."""
    D = pdist(X.astype(np.float64), metric="cosine")
    np.clip(D, 0.0, None, out=D)      # guard tiny negative rounding artifacts
    Z = linkage(D, method="average")
    return {"path": "exact", "Z": Z, "n_leaves": int(X.shape[0]),
            "micro_labels": None, "centroids": None}


def build_hierarchy_large(X, seed=None):
    """APPROXIMATION: micro-cluster with MiniBatchKMeans, then exact hierarchy
    over the (re-normalized) micro-centroids. Rows inherit their centroid's cut."""
    seed = RANDOM_STATE if seed is None else seed
    n = X.shape[0]
    n_micro = int(min(N_MICRO_MAX, max(50, n // 10)))
    km = MiniBatchKMeans(n_clusters=n_micro, random_state=seed,
                         batch_size=4096, n_init="auto")
    ml = km.fit_predict(X)
    present = np.unique(ml)                       # MiniBatchKMeans may leave empties
    remap = np.full(n_micro, -1, dtype=int)
    remap[present] = np.arange(len(present))
    C = km.cluster_centers_[present].astype(np.float64)
    norms = np.linalg.norm(C, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    C = C / norms
    Z = linkage(pdist(C, metric="cosine"), method="average")
    return {"path": "micro-approx", "Z": Z, "n_leaves": int(len(present)),
            "micro_labels": remap[ml], "centroids": C}


def build_hierarchy(X):
    if X.shape[0] > LARGE_HS_THRESHOLD:
        return build_hierarchy_large(X)
    return build_hierarchy_small(X)


def cut_tree_to_groups(hier, k):
    """Row-level labels (1..m, m <= k) for one cut of the hierarchy."""
    leaf = fcluster(hier["Z"], t=k, criterion="maxclust")
    if hier["micro_labels"] is None:
        return leaf
    return leaf[hier["micro_labels"]]

# ---- caching ----------------------------------------------------------------

def _fp(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True,
                                     default=str).encode()).hexdigest()[:12]


def data_fingerprint(hs, meta):
    pid_hash = int(pd.util.hash_pandas_object(meta["point_id"].astype(str),
                                              index=False).sum() % (2 ** 63))
    return _fp({"hs": hs, "n": len(meta), "pid": pid_hash})


def tree_cfg():
    return {"LARGE_HS_THRESHOLD": LARGE_HS_THRESHOLD, "N_MICRO_MAX": N_MICRO_MAX,
            "RANDOM_STATE": RANDOM_STATE}


def sweep_cfg():
    return {**tree_cfg(), "K_MIN": K_MIN, "K_MAX": K_MAX,
            "MIN_GROUP_CNT": MIN_GROUP_CNT, "MIN_COVERAGE": MIN_COVERAGE,
            "ETA_TOL": ETA_TOL, "TARGET_SPREAD": TARGET_SPREAD,
            "MIN_SPREAD_N": MIN_SPREAD_N, "SILHOUETTE_SAMPLE": SILHOUETTE_SAMPLE}


def get_hierarchy(hs, meta, X):
    fp = _fp({"data": data_fingerprint(hs, meta), **tree_cfg()})
    path = CACHE_PATH / f"hs10_{hs}_tree_{fp}.pkl"
    if USE_CACHE and path.exists():
        with open(path, "rb") as fh:
            hier = pickle.load(fh)
        print(f"Loaded cached clustering result for HS10 {hs} "
              f"({hier['path']}, tree {fp})")
        return hier
    t0 = time.time()
    hier = build_hierarchy(X)
    hier["build_seconds"] = round(time.time() - t0, 2)
    if USE_CACHE:
        with open(path, "wb") as fh:
            pickle.dump(hier, fh)
    print(f"Computed{' and cached' if USE_CACHE else ''} clustering hierarchy for "
          f"HS10 {hs} ({hier['path']}, {hier['n_leaves']:,} leaves, "
          f"{hier['build_seconds']:.1f}s, tree {fp})")
    return hier

# ---- silhouette (DIAGNOSTIC ONLY — never used for k-selection/classification) --

def prep_silhouette(X, seed=None):
    seed = RANDOM_STATE if seed is None else seed
    n = X.shape[0]
    m = int(min(SILHOUETTE_SAMPLE, n))
    idx = np.sort(np.random.default_rng(seed).choice(n, m, replace=False))
    D = squareform(pdist(X[idx].astype(np.float64), metric="cosine"))
    return idx, D


def silhouette_at(D, labels_sample):
    labs = np.asarray(labels_sample)
    if len(np.unique(labs)) < 2 or len(np.unique(labs)) >= len(labs):
        return np.nan
    try:
        return float(silhouette_score(D, labs, metric="precomputed"))
    except ValueError:
        return np.nan

print("Clustering + caching functions defined.")

In [ ]:
# Build (or load from cache) the hierarchy for the walkthrough HS10.
wt_hier = get_hierarchy(wt_hs, wt_meta, wt_X)
print(f"method   : {wt_hier['path']}"
      + ("  <- APPROXIMATION (see the comparison cell below)"
         if wt_hier["path"] == "micro-approx" else "  (reference method)"))
print(f"leaves   : {wt_hier['n_leaves']:,} "
      + ("micro-cluster centroids" if wt_hier["path"] == "micro-approx" else "rows"))
print(f"max k    : {min(K_MAX, wt_hier['n_leaves'] - 1)} (K_MAX={K_MAX}, "
      f"capped at leaves-1)")
_h = wt_hier["Z"][:, 2]
print(f"merge heights (cosine distance): min {_h.min():.4f}, "
      f"median {np.median(_h):.4f}, max {_h.max():.4f}")

In [ ]:
# Approximation-quality check (only meaningful when the micro path is in use).
# On ONE shared sample: exact hierarchy vs micro-cluster approximation, same k cuts.
# Reports adjusted Rand index (label agreement) and side-by-side price metrics, so
# the micro-approx result is never silently treated as the exact result.

def compare_exact_vs_micro(X, frame, ks, seed=None):
    seed = RANDOM_STATE if seed is None else seed
    n = X.shape[0]
    m = int(min(APPROX_COMPARE_SAMPLE, n))
    idx = np.sort(np.random.default_rng(seed).choice(n, m, replace=False))
    Xs = X[idx]
    fs = frame.iloc[idx].reset_index(drop=True)
    exact = build_hierarchy_small(Xs)
    micro = build_hierarchy_large(Xs, seed)
    rows = []
    for k in ks:
        k = int(min(k, exact["n_leaves"] - 1, micro["n_leaves"] - 1))
        if k < 2:
            continue
        le = cut_tree_to_groups(exact, k)
        lm = cut_tree_to_groups(micro, k)
        me = economic_metrics(fs, le)
        mm = economic_metrics(fs, lm)
        rows.append({
            "k": k, "sample_n": m,
            "ARI": float(adjusted_rand_score(le, lm)),
            "spread_exact": me["wm_spread"], "spread_micro": mm["wm_spread"],
            "eta2_exact": me["eta2"], "eta2_micro": mm["eta2"],
            "coverage_exact": me["coverage"], "coverage_micro": mm["coverage"],
        })
    return pd.DataFrame(rows)


def report_approx_comparison(cmp_df, hs):
    display(cmp_df.round(4))
    ari = float(cmp_df["ARI"].min())
    d_spread = float(np.nanmax(np.abs(cmp_df["spread_exact"] - cmp_df["spread_micro"])))
    if ari >= APPROX_ARI_GOOD and (not np.isfinite(d_spread) or d_spread <= 0.05):
        verdict = "approximation agrees closely with the exact method on this sample"
    elif ari >= 0.40:
        verdict = ("moderate agreement — micro-approx groupings differ somewhat from "
                   "the exact method; treat fine-grained group boundaries with caution")
    else:
        verdict = ("MATERIALLY DIFFERENT — the micro-cluster approximation does not "
                   "reproduce the exact clustering here; its results for this HS10 "
                   "should not be read as the reference method's")
    print(f"HS10 {hs}: min ARI {ari:.3f}, max |spread difference| {d_spread:.4f} "
          f"-> {verdict}.")

wt_approx = None
if wt_hier["path"] == "micro-approx" and RUN_APPROX_COMPARISON:
    print("NOTE: economic_metrics() is defined two cells below; if this cell errors "
          "on the first pass, run the sweep-function cell and re-run this one.")
elif wt_hier["path"] == "exact":
    print(f"HS10 {wt_hs} used the exact reference method "
          f"(n = {len(wt_meta):,} <= LARGE_HS_THRESHOLD = {LARGE_HS_THRESHOLD:,}) — "
          f"no approximation to check here. The check runs automatically in the "
          f"batch pipeline for any HS10 on the micro path.")

In [ ]:
# k-sweep: for every k, category-A (embedding-only) and category-B (price) metrics.

def clustering_metrics(labels):
    """Category A — computed from group sizes only; NO price involved."""
    sizes = pd.Series(labels).value_counts().to_numpy()
    return {
        "n_groups": int(sizes.size),
        "size_min": int(sizes.min()),
        "size_median": float(np.median(sizes)),
        "size_p10": float(np.quantile(sizes, 0.10)),
        "tiny_group_share": float((sizes < MIN_GROUP_CNT).mean()),
    }


def economic_metrics(frame, labels):
    """Category B — price metrics computed strictly AFTER clustering."""
    w = frame["OBS_WEIGHT"].to_numpy(dtype=float)
    y = frame["LOG_PRICE"].to_numpy(dtype=float)
    v = frame["PRICE_VALID"].to_numpy(dtype=bool)
    lab = np.asarray(labels)
    sizes = pd.Series(lab).value_counts()
    wsum = pd.Series(w).groupby(lab).sum()

    spreads = {}
    dfv = pd.DataFrame({"g": lab[v], "y": y[v], "w": w[v]})
    for g, sub in dfv.groupby("g"):
        spreads[g] = robust_spread(sub["y"].to_numpy(), sub["w"].to_numpy())
    spreads = pd.Series(spreads, dtype=float)

    big = sizes.index[sizes >= MIN_GROUP_CNT]
    qual = [g for g in big if g in spreads.index and np.isfinite(spreads[g])]
    if qual:
        sv = spreads[qual].to_numpy()
        wv = wsum[qual].to_numpy()
        wm_spread = float(np.average(sv, weights=wv))
        wmed_spread, p90_spread = (float(x) for x in
                                   weighted_quantile(sv, [0.5, 0.9], wv))
    else:
        wm_spread = wmed_spread = p90_spread = np.nan

    coverage = float(w[np.isin(lab, np.asarray(big))].sum() / w.sum())
    eta2, W, k_data = weighted_eta_squared(y[v], w[v], lab[v])
    adj = adjusted_eta_squared(eta2, W, k_data)
    good = spreads.index[spreads.to_numpy() <= TARGET_SPREAD] if len(spreads) else []
    in_good = np.isin(lab, np.asarray(good)) & v
    wv_sum = w[v].sum()
    share_le_target = float(w[in_good].sum() / wv_sum) if wv_sum > 0 else np.nan

    return {"eta2": eta2, "adj_eta2": adj, "wm_spread": wm_spread,
            "wmed_spread": wmed_spread, "p90_spread": p90_spread,
            "coverage": coverage, "share_spread_le_target": share_le_target,
            "n_qual_groups": int(len(qual))}


def sweep_k(hier, frame, X):
    K_hi = int(min(K_MAX, hier["n_leaves"] - 1))
    if K_hi < K_MIN:
        raise ValueError(f"only {hier['n_leaves']} leaves — cannot sweep k >= {K_MIN}")
    sil_idx, sil_D = prep_silhouette(X)
    rows = []
    for k in range(K_MIN, K_hi + 1):
        lab = cut_tree_to_groups(hier, k)
        rows.append({"k": k, **clustering_metrics(lab),
                     "silhouette": silhouette_at(sil_D, lab[sil_idx]),
                     **economic_metrics(frame, lab)})
    del sil_D
    return pd.DataFrame(rows)


def get_sweep(hs, meta, frame, X, hier):
    fp = _fp({"data": data_fingerprint(hs, meta), **sweep_cfg()})
    path = CACHE_PATH / f"hs10_{hs}_k_results_{fp}.parquet"
    if USE_CACHE and path.exists():
        print(f"Loaded cached k-sweep for HS10 {hs} (sweep {fp})")
        return pd.read_parquet(path)
    t0 = time.time()
    sweep = sweep_k(hier, frame, X)
    if USE_CACHE:
        sweep.to_parquet(path)
    print(f"Computed{' and cached' if USE_CACHE else ''} k-sweep for HS10 {hs} "
          f"({len(sweep)} cuts, {time.time() - t0:.1f}s, sweep {fp})")
    return sweep

print("k-sweep functions defined.")

In [ ]:
# Run the k-sweep for the walkthrough HS10 and show the full results table.
wt_sweep = get_sweep(wt_hs, wt_meta, wt, wt_X, wt_hier)

# If the walkthrough HS10 uses the micro-approx path, run the deferred
# approximation-quality check now that economic_metrics exists.
if wt_hier["path"] == "micro-approx" and RUN_APPROX_COMPARISON and wt_approx is None:
    _ks = sorted({K_MIN + (min(K_MAX, wt_hier["n_leaves"] - 1) - K_MIN) // 2,
                  min(K_MAX, wt_hier["n_leaves"] - 1)})
    wt_approx = compare_exact_vs_micro(wt_X, wt, _ks)
    report_approx_comparison(wt_approx, wt_hs)

with pd.option_context("display.max_rows", 200):
    display(wt_sweep.round(4))

In [ ]:
# k-selection: coverage-gated, adjusted-eta^2 within tolerance, PREFER FEWER GROUPS.
# Inputs: coverage + adjusted eta^2 + group-size sanity. Silhouette is NOT an input.

def select_k(sweep):
    K_hi = int(sweep["k"].max())
    cand = sweep[sweep["coverage"] >= MIN_COVERAGE]
    low_coverage = cand.empty
    if low_coverage:
        cand = sweep.loc[[sweep["coverage"].idxmax()]]
    if cand["adj_eta2"].notna().any():
        best = float(cand["adj_eta2"].max())
        ok = cand[cand["adj_eta2"] >= best - ETA_TOL]
        chosen_k = int(ok["k"].min())
        best_k = int(cand.loc[cand["adj_eta2"].idxmax(), "k"])
    else:
        best, chosen_k = np.nan, int(cand["k"].min())
        best_k = chosen_k
    return {"chosen_k": chosen_k, "best_k": best_k, "best_adj_eta2": best,
            "low_coverage": low_coverage, "K_hi": K_hi,
            "max_k_reached": bool(chosen_k >= K_hi or best_k >= K_hi),
            "n_candidates": int(len(cand))}

wt_sel = select_k(wt_sweep)
_chosen_row = wt_sweep.loc[wt_sweep["k"] == wt_sel["chosen_k"]].iloc[0]
print(f"HS10 {wt_hs} — k selection")
print(f"  candidate cuts with coverage >= {MIN_COVERAGE:.0%} : "
      f"{wt_sel['n_candidates']} of {len(wt_sweep)}"
      + ("  (NONE met the gate — fell back to the max-coverage cut)"
         if wt_sel["low_coverage"] else ""))
print(f"  best adjusted eta^2                    : "
      f"{wt_sel['best_adj_eta2']:.4f} at k={wt_sel['best_k']}")
print(f"  CHOSEN k (smallest within ETA_TOL={ETA_TOL}) : {wt_sel['chosen_k']}")
print(f"  at chosen k: coverage {_chosen_row['coverage']:.1%}, "
      f"grouped spread {_chosen_row['wm_spread']:.3f}, "
      f"eta^2 {_chosen_row['eta2']:.4f}")
print(f"  selection pushed against K_MAX          : {wt_sel['max_k_reached']}")

In [ ]:
# k vs eta^2 / adjusted eta^2, with the chosen k marked.
def plot_k_eta(sweep, sel, hs, ax=None):
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(7.5, 3.2))
    ax.plot(sweep["k"], sweep["eta2"], label="eta²", color=PALETTE[0])
    ax.plot(sweep["k"], sweep["adj_eta2"], label="adjusted eta²", color=PALETTE[1])
    ax.axvline(sel["chosen_k"], color=INK2, linestyle="--", linewidth=1.2)
    ax.annotate(f"chosen k = {sel['chosen_k']}",
                xy=(sel["chosen_k"], ax.get_ylim()[1]),
                xytext=(4, -2), textcoords="offset points",
                va="top", fontsize=8, color=INK2)
    ax.set_xlabel("k (number of groups)")
    ax.set_ylabel("explained log-price variance")
    if standalone:
        ax.set_title(f"HS10 {hs} — price variance explained by description groups")
    else:
        ax.set_title(f"k-selection (chosen k = {sel['chosen_k']})")
    ax.legend(loc="lower right", fontsize=8)
    if standalone:
        plt.show()

plot_k_eta(wt_sweep, wt_sel, wt_hs)

In [ ]:
# k vs usable coverage and k vs weighted within-group spread (separate panels).
def plot_k_coverage_spread(sweep, sel, hs, base=None, axes=None):
    standalone = axes is None
    if standalone:
        fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.2))
    ax = axes[0]
    ax.plot(sweep["k"], sweep["coverage"], color=PALETTE[0])
    ax.axhline(MIN_COVERAGE, color=INK2, linewidth=1.0, linestyle=":")
    ax.annotate(f"MIN_COVERAGE {MIN_COVERAGE:.0%}", xy=(sweep["k"].min(),
                MIN_COVERAGE), xytext=(2, 3), textcoords="offset points",
                fontsize=8, color=INK2)
    ax.axvline(sel["chosen_k"], color=INK2, linestyle="--", linewidth=1.2)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("k")
    ax.set_ylabel("usable coverage (weighted)")
    ax.set_title(f"HS10 {hs} — coverage vs k")

    ax = axes[1]
    ax.plot(sweep["k"], sweep["wm_spread"], color=PALETTE[0])
    ax.axhline(TARGET_SPREAD, color=INK2, linewidth=1.0, linestyle=":")
    ax.annotate(f"target {TARGET_SPREAD}", xy=(sweep["k"].min(), TARGET_SPREAD),
                xytext=(2, 3), textcoords="offset points", fontsize=8, color=INK2)
    if base is not None and np.isfinite(base.get("baseline_spread", np.nan)):
        ax.axhline(base["baseline_spread"], color=MUTED, linewidth=1.0,
                   linestyle="--")
        ax.annotate(f"baseline {base['baseline_spread']:.2f}",
                    xy=(sweep["k"].max(), base["baseline_spread"]),
                    xytext=(-2, 3), textcoords="offset points", ha="right",
                    fontsize=8, color=MUTED)
    ax.axvline(sel["chosen_k"], color=INK2, linestyle="--", linewidth=1.2)
    ax.set_xlabel("k")
    ax.set_ylabel("weighted within-group spread")
    ax.set_title(f"HS10 {hs} — dispersion vs k")
    if standalone:
        plt.show()

plot_k_coverage_spread(wt_sweep, wt_sel, wt_hs, wt_base)

In [ ]:
# Cut the hierarchy at the chosen k and assign human-readable group IDs.
def assign_group_ids(hs, labels):
    """Group IDs numbered by descending size: <HS>-G01 is the largest group."""
    sizes = pd.Series(labels).value_counts()
    mapping = {g: f"{hs}-G{r + 1:02d}" for r, g in enumerate(sizes.index)}
    return pd.Series(labels).map(mapping).to_numpy(), mapping


def get_group_ids(hs, meta, frame, hier, chosen_k):
    fp = _fp({"data": data_fingerprint(hs, meta), **sweep_cfg()})
    path = CACHE_PATH / f"hs10_{hs}_groups_{fp}.parquet"
    if USE_CACHE and path.exists():
        cached = pd.read_parquet(path)
        if len(cached) == len(frame):
            print(f"Loaded cached group assignment for HS10 {hs} (groups {fp})")
            lookup = dict(zip(cached["point_id"], cached["GROUP_ID"]))
            return frame["point_id"].map(lookup).to_numpy()
    labels = cut_tree_to_groups(hier, chosen_k)
    gids, _ = assign_group_ids(hs, labels)
    if USE_CACHE:
        pd.DataFrame({"point_id": frame["point_id"].to_numpy(),
                      "GROUP_ID": gids}).to_parquet(path)
    return gids

wt = wt.copy()
wt["GROUP_ID"] = get_group_ids(wt_hs, wt_meta, wt, wt_hier, wt_sel["chosen_k"])
wt_sizes = wt["GROUP_ID"].value_counts()
print(f"HS10 {wt_hs}: {len(wt_sizes)} commodity groups at k={wt_sel['chosen_k']}")
display(wt[["point_id", "GROUP_ID", "ACCURATE_NAME", "PRICE_KG",
            "MANUFACTURE_COUNTRY_CODE", "TRANSPORT"]].head(15))

In [ ]:
# Group-size distribution and over-fragmentation report.
def overfragmentation_report(sel, group_sizes):
    sizes = np.asarray(group_sizes, dtype=float)
    rep = {
        "chosen_k": sel["chosen_k"],
        "max_k_reached": sel["max_k_reached"],
        "min_group_size": int(sizes.min()),
        "median_group_size": float(np.median(sizes)),
        "p10_group_size": float(np.quantile(sizes, 0.10)),
        "small_group_share": float((sizes < MIN_GROUP_CNT).mean()),
    }
    rep["OVERFRAGMENTED"] = bool(
        rep["small_group_share"] > MAX_SMALL_GROUP_SHARE
        or rep["median_group_size"] < MIN_GROUP_CNT
        or rep["max_k_reached"])
    return rep


def plot_group_sizes(group_sizes, hs, ax=None):
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(7.5, 3.0))
    sizes = np.sort(np.asarray(group_sizes))[::-1]
    ax.bar(np.arange(1, len(sizes) + 1), sizes, color=PALETTE[0],
           edgecolor=SURFACE, linewidth=0.4)
    ax.axhline(MIN_GROUP_CNT, color=INK2, linewidth=1.0, linestyle=":")
    ax.annotate(f"MIN_GROUP_CNT = {MIN_GROUP_CNT}",
                xy=(0.02, MIN_GROUP_CNT), xycoords=("axes fraction", "data"),
                xytext=(0, 3), textcoords="offset points", ha="left",
                fontsize=8, color=INK2,
                bbox={"facecolor": SURFACE, "edgecolor": "none", "alpha": 0.8,
                      "pad": 1.5})
    if sizes.max() / max(sizes.min(), 1) > 50:
        ax.set_yscale("log")
    ax.set_xlabel("group rank (largest first)")
    ax.set_ylabel("rows in group")
    if standalone:
        ax.set_title(f"HS10 {hs} — group-size distribution")
    else:
        ax.set_title("group-size distribution")
    if standalone:
        plt.show()

wt_ofr = overfragmentation_report(wt_sel, wt_sizes.to_numpy())
plot_group_sizes(wt_sizes.to_numpy(), wt_hs)
for k_, v_ in wt_ofr.items():
    print(f"  {k_:<22}: {v_}")
if wt_ofr["OVERFRAGMENTED"]:
    print("  -> flagged OVERFRAGMENTED (dominated by tiny groups and/or the optimum "
          "pushes against K_MAX). Do not raise K_MAX just to chase lower dispersion.")

In [ ]:
# Per-group price profile; flag MIXED groups. Show the largest and the widest.
def group_profile(frame):
    rows = []
    for gid, sub in frame.groupby("GROUP_ID"):
        v = sub["PRICE_VALID"].to_numpy()
        w = sub["OBS_WEIGHT"].to_numpy(dtype=float)
        p = sub["PRICE_KG_NUM"].to_numpy(dtype=float)
        qs = weighted_quantile(p[v], [0.10, 0.25, 0.50, 0.75, 0.90], w[v])
        spread = robust_spread(sub["LOG_PRICE"].to_numpy()[v], w[v])
        rows.append({"GROUP_ID": gid, "N": int(len(sub)),
                     "W": round(float(w.sum()), 1),
                     "PRICE_P10": qs[0], "PRICE_P25": qs[1], "PRICE_P50": qs[2],
                     "PRICE_P75": qs[3], "PRICE_P90": qs[4],
                     "ROBUST_SPREAD": spread,
                     "MIXED": bool(np.isfinite(spread)
                                   and spread > MIXED_SPREAD_FACTOR * TARGET_SPREAD)})
    return (pd.DataFrame(rows)
            .sort_values("N", ascending=False, ignore_index=True))

wt_profile = group_profile(wt)
print(f"Largest groups (of {len(wt_profile)}):")
display(wt_profile.head(TOP_GROUPS_DISPLAY).round(3))
_wide = wt_profile[np.isfinite(wt_profile["ROBUST_SPREAD"])].sort_values(
    "ROBUST_SPREAD", ascending=False)
print("Widest groups (highest robust spread):")
display(_wide.head(5).round(3))
_n_mixed = int(wt_profile["MIXED"].sum())
print(f"{_n_mixed} group(s) flagged MIXED "
      f"(spread > {MIXED_SPREAD_FACTOR} x {TARGET_SPREAD} = "
      f"{MIXED_SPREAD_FACTOR * TARGET_SPREAD:.3f}).")
print("CAVEAT: high price dispersion does NOT automatically mean the descriptions "
      "are semantically mixed — a genuine commodity can have heterogeneous prices. "
      "Judge the representative descriptions in the next cell before concluding.")

In [ ]:
# Representative descriptions per group: members NEAREST TO THE GROUP CENTROID
# in cosine similarity (not the first rows of the table).
def representative_names(frame, X, group_id, n=None):
    n = N_REPRESENTATIVES if n is None else n
    idx = np.flatnonzero((frame["GROUP_ID"] == group_id).to_numpy())
    centroid = X[idx].mean(axis=0)
    nrm = np.linalg.norm(centroid)
    if nrm > 0:
        centroid = centroid / nrm
    sims = X[idx] @ centroid
    names = frame["ACCURATE_NAME"].to_numpy()
    out, seen = [], set()
    for j in np.argsort(-sims):
        nm = str(names[idx[j]])
        if nm in seen:
            continue
        seen.add(nm)
        out.append({"cos_sim": round(float(sims[j]), 4), "ACCURATE_NAME": nm})
        if len(out) >= n:
            break
    return pd.DataFrame(out)


def show_group_representatives(frame, X, profile, group_ids, title):
    print(title)
    prof = profile.set_index("GROUP_ID")
    for gid in group_ids:
        r = prof.loc[gid]
        spread_txt = (f"{r['ROBUST_SPREAD']:.3f}"
                      if np.isfinite(r["ROBUST_SPREAD"]) else "n/a")
        print(f"\n### {gid}  |  N={int(r['N'])}  |  median price "
              f"{r['PRICE_P50']:.2f}/kg  |  robust spread {spread_txt}"
              + ("  |  MIXED" if r["MIXED"] else ""))
        display(representative_names(frame, X, gid))

_top_ids = wt_profile["GROUP_ID"].head(TOP_GROUPS_DISPLAY).tolist()
show_group_representatives(wt, wt_X, wt_profile, _top_ids,
                           f"HS10 {wt_hs} — top {len(_top_ids)} groups by size")

_wide_ids = [g for g in wt_profile[wt_profile["MIXED"]]["GROUP_ID"]
             if g not in _top_ids][:5]
if _wide_ids:
    show_group_representatives(
        wt, wt_X, wt_profile, _wide_ids,
        f"HS10 {wt_hs} — MIXED-flagged groups (inspect: one commodity or several?)")

In [ ]:
# Grouped price distributions: baseline vs the largest groups.
def plot_group_boxes(frame, hs, profile, top_n=None, ax=None):
    top_n = BOX_TOP_GROUPS if top_n is None else top_n
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(9.5, 3.6))
    gids = profile["GROUP_ID"].head(top_n).tolist()
    series = [frame.loc[frame["PRICE_VALID"], "LOG_PRICE"].to_numpy()]
    labels = ["ALL"]
    for g in gids:
        m = (frame["GROUP_ID"] == g) & frame["PRICE_VALID"]
        series.append(frame.loc[m, "LOG_PRICE"].to_numpy())
        labels.append(g.split("-")[-1])
    bp = ax.boxplot(series, tick_labels=labels, showfliers=False, patch_artist=True,
                    medianprops={"color": INK, "linewidth": 1.2},
                    whiskerprops={"color": AXIS_C}, capprops={"color": AXIS_C})
    for i, box in enumerate(bp["boxes"]):
        box.set(facecolor=(AXIS_C if i == 0 else PALETTE[0]),
                alpha=0.55, edgecolor=INK2, linewidth=0.8)
    ax.set_ylabel("ln(PRICE_KG)")
    ax.set_xlabel("commodity group (largest first; boxes show unweighted rows)")
    if standalone:
        ax.set_title(f"HS10 {hs} — price distributions: baseline vs description "
                     f"groups")
    else:
        ax.set_title("prices: ALL vs description groups")
    ax.tick_params(axis="x", labelsize=8)
    if standalone:
        plt.show()

plot_group_boxes(wt, wt_hs, wt_profile)
print("Boxes summarize raw rows for readability; all reported metrics use the "
      "line-level weights.")

In [ ]:
# Silhouette diagnostic at the chosen k (embedding geometry only).
_row = wt_sweep.loc[wt_sweep["k"] == wt_sel["chosen_k"]].iloc[0]
print(f"HS10 {wt_hs} at k={wt_sel['chosen_k']}: cosine silhouette = "
      f"{_row['silhouette']:.4f} "
      f"(sample of {min(SILHOUETTE_SAMPLE, len(wt)):,} rows)")
print("Reminder: silhouette describes how separated the description embeddings are. "
      "It is a DIAGNOSTIC only — it does not influence k-selection, classification, "
      "or membership, and a low silhouette does not preclude economically useful "
      "groups (nor does a high one guarantee them).")

In [ ]:
# Classify the walkthrough HS10 and print the full before/after comparison.
def classify_hs(m):
    """Ordered decision rules; every threshold comes from the configuration cell."""
    if np.isfinite(m["baseline_spread"]) and m["baseline_spread"] <= TARGET_SPREAD:
        return "NEUTRAL", ("baseline spread is already at/below the target — this "
                           "HS10 is homogeneous without description grouping")
    if (m["small_group_share"] > MAX_SMALL_GROUP_SHARE
            or m["coverage"] < MIN_COVERAGE
            or m["median_group_size"] < MIN_GROUP_CNT):
        return "OVERFRAGMENTED", ("the solution is dominated by tiny groups or "
                                  "leaves too many observations outside usable "
                                  "groups")
    if (np.isfinite(m["reduction"]) and m["reduction"] >= STRONG_REDUCTION
            and m["coverage"] >= MIN_COVERAGE):
        return "STRONG_SUCCESS", ("description grouping substantially reduces price "
                                  "dispersion with reasonably sized groups and "
                                  "sufficient coverage")
    if (np.isfinite(m["reduction"]) and m["reduction"] >= USEFUL_REDUCTION
            and m["coverage"] >= MIN_COVERAGE):
        return "USEFUL", ("meaningful improvement at sufficient coverage, but "
                          "heterogeneity is not fully resolved")
    return "INSUFFICIENT_DESCRIPTION", (
        "price remains heterogeneous after grouping — the descriptions (as embedded) "
        "do not carry enough price-relevant information for this HS10")


def hs_metrics(base, sweep, sel, ofr):
    row = sweep.loc[sweep["k"] == sel["chosen_k"]].iloc[0]
    b, g = base["baseline_spread"], row["wm_spread"]
    reduction = (1.0 - g / b) if (np.isfinite(b) and b > 0
                                  and np.isfinite(g)) else np.nan
    return {"baseline_spread": b, "grouped_spread": float(g)
            if np.isfinite(g) else np.nan, "reduction": reduction,
            "eta2": float(row["eta2"]), "adj_eta2": float(row["adj_eta2"]),
            "coverage": float(row["coverage"]),
            "share_spread_le_target": float(row["share_spread_le_target"]),
            "small_group_share": ofr["small_group_share"],
            "median_group_size": ofr["median_group_size"],
            "min_group_size": ofr["min_group_size"]}

wt_metrics = hs_metrics(wt_base, wt_sweep, wt_sel, wt_ofr)
wt_status, wt_reason = classify_hs(wt_metrics)

print(f"HS10 {wt_hs}")
print(f"  BASELINE   N = {wt_base['n_rows']:,} | robust spread "
      f"{wt_metrics['baseline_spread']:.3f} | usable coverage "
      f"{wt_base['baseline_coverage']:.1%}")
print(f"  GROUPED    k = {wt_sel['chosen_k']} | robust within-group spread "
      f"{wt_metrics['grouped_spread']:.3f} | usable coverage "
      f"{wt_metrics['coverage']:.1%} | eta^2 {wt_metrics['eta2']:.3f}")
print(f"  dispersion_reduction = 1 - grouped/baseline = "
      f"{wt_metrics['reduction']:.1%}")
print(f"  share of observations in groups at/below target spread: "
      f"{wt_metrics['share_spread_le_target']:.1%}")
print(f"\n  STATUS: {wt_status} — {wt_reason}.")
print("\n  (Numbers alone do not settle it: confirm against the representative "
      "descriptions above before trusting the label.)")

## Batch analysis

The cells below run the *identical* pipeline (weights → baseline → hierarchy →
k-sweep → k-selection → groups → profiles → classification) for **every loaded
HS10**, reusing the caches, then build the master evaluation table, per-HS10
diagnostic panels, the cross-HS10 comparison, and the final exports.

In [ ]:
# The full per-HS10 pipeline as one function (composes the walkthrough steps).
def run_pipeline_for_hs(hs, meta, X):
    frame = prepare_frame(meta)
    base = baseline_stats(frame)
    hier = get_hierarchy(hs, meta, X)
    sweep = get_sweep(hs, meta, frame, X, hier)
    sel = select_k(sweep)
    frame = frame.copy()
    frame["GROUP_ID"] = get_group_ids(hs, meta, frame, hier, sel["chosen_k"])
    sizes = frame["GROUP_ID"].value_counts()
    ofr = overfragmentation_report(sel, sizes.to_numpy())
    profile = group_profile(frame)
    metrics = hs_metrics(base, sweep, sel, ofr)
    status, reason = classify_hs(metrics)

    approx = None
    if hier["path"] == "micro-approx" and RUN_APPROX_COMPARISON:
        ks = sorted({sel["chosen_k"], min(sel["K_hi"], 2 * sel["chosen_k"])})
        approx = compare_exact_vs_micro(X, frame, ks)

    eval_row = {
        "HS_CODE": hs, "N": base["n_rows"],
        "BASELINE_SPREAD": metrics["baseline_spread"],
        "GROUPED_SPREAD": metrics["grouped_spread"],
        "DISPERSION_REDUCTION": metrics["reduction"],
        "ETA_SQUARED": metrics["eta2"],
        "BASELINE_USABLE_COVERAGE": base["baseline_coverage"],
        "GROUPED_USABLE_COVERAGE": metrics["coverage"],
        "CHOSEN_K": sel["chosen_k"],
        "MEDIAN_GROUP_SIZE": metrics["median_group_size"],
        "MIN_GROUP_SIZE": metrics["min_group_size"],
        "SMALL_GROUP_SHARE": metrics["small_group_share"],
        "STATUS": status,
        "METHOD": hier["path"],
        "APPROX_ARI": (float(approx["ARI"].min())
                       if approx is not None and len(approx) else np.nan),
    }
    groups_df = frame[["HS_CODE", "GROUP_ID", "point_id", "DECL_ID", "CMDT_ID",
                       "ACCURATE_NAME", "PRICE_KG", "MANUFACTURE_COUNTRY_CODE",
                       "TRANSPORT"]].copy()
    groups_df["GROUP_SIZE"] = groups_df["GROUP_ID"].map(sizes).astype(int)
    diag = {"frame": frame, "base": base, "sweep": sweep, "sel": sel, "ofr": ofr,
            "profile": profile, "metrics": metrics, "status": status,
            "reason": reason, "method": hier["path"], "approx": approx}
    return eval_row, groups_df, diag

print("Pipeline function defined.")

In [ ]:
# Run the pipeline for every loaded HS10 and build the master evaluation table.
batch_order = ([h for h in SELECTED_HS_CODES if h in data]
               + [h for h in data if h not in SELECTED_HS_CODES])
eval_rows, all_groups, batch_diag = [], [], {}
for hs in batch_order:
    print(f"\n=== HS10 {hs} ({len(data[hs]['meta']):,} rows) ===")
    try:
        row, gdf, diag = run_pipeline_for_hs(hs, data[hs]["meta"], data[hs]["X"])
    except ValueError as e:
        print(f"  SKIPPED: {e}")
        continue
    eval_rows.append(row)
    all_groups.append(gdf)
    batch_diag[hs] = diag
    if diag["approx"] is not None:
        report_approx_comparison(diag["approx"], hs)
    print(f"  {row['STATUS']}: baseline {row['BASELINE_SPREAD']:.3f} -> grouped "
          f"{row['GROUPED_SPREAD']:.3f} (reduction "
          f"{row['DISPERSION_REDUCTION']:.1%}) at k={row['CHOSEN_K']}, coverage "
          f"{row['GROUPED_USABLE_COVERAGE']:.1%} [{row['METHOD']}]")

master = (pd.DataFrame(eval_rows)
          .sort_values("DISPERSION_REDUCTION", ascending=False, ignore_index=True))
print("\nMaster evaluation table (micro-approx rows are approximations — see the "
      "METHOD column and their ARI checks above):")
display(master.round(4))

In [ ]:
# Per-HS10 diagnostic panels: baseline vs grouped prices, k-curve, group sizes,
# representative descriptions, largest/widest groups, and a short conclusion.
for hs in [h for h in batch_order if h in batch_diag]:
    diag = batch_diag[hs]
    frame, profile = diag["frame"], diag["profile"]
    fig, axes = plt.subplots(2, 2, figsize=(11, 6.4))
    plot_baseline_hist(frame, hs, diag["base"], ax=axes[0, 0])
    plot_group_boxes(frame, hs, profile, top_n=8, ax=axes[0, 1])
    plot_k_eta(diag["sweep"], diag["sel"], hs, ax=axes[1, 0])
    plot_group_sizes(frame["GROUP_ID"].value_counts().to_numpy(), hs, ax=axes[1, 1])
    fig.suptitle(f"HS10 {hs} — {diag['status']} ({diag['method']})", y=1.0)
    plt.show()

    _tops = profile["GROUP_ID"].head(5).tolist()
    show_group_representatives(frame, data[hs]["X"], profile, _tops,
                               f"HS10 {hs} — top 5 groups")
    _wide = profile[np.isfinite(profile["ROBUST_SPREAD"])].sort_values(
        "ROBUST_SPREAD", ascending=False)
    _wide_ids = [g for g in _wide["GROUP_ID"].head(2) if g not in _tops]
    if _wide_ids:
        show_group_representatives(frame, data[hs]["X"], profile, _wide_ids,
                                   f"HS10 {hs} — widest groups")
    m = diag["metrics"]
    print(f"\nConclusion for HS10 {hs}: {diag['status']} — {diag['reason']}. "
          f"Baseline spread {m['baseline_spread']:.3f} -> grouped "
          f"{m['grouped_spread']:.3f} (reduction {m['reduction']:.1%}) at "
          f"k={diag['sel']['chosen_k']} with coverage {m['coverage']:.1%}.")
    print("=" * 100)

In [ ]:
# Cross-HS10 comparison: successful vs neutral vs difficult codes.
EXPECTED_PROFILE = {
    "6109100000": "expected success", "6115950000": "expected success",
    "6111209000": "expected success", "2005202000": "neutral / control",
    "4016930005": "difficult", "3304990000": "difficult",
    "3926909709": "difficult", "7326909807": "difficult",
    "8708299001": "difficult", "8708999701": "difficult",
}

_m = master.sort_values("DISPERSION_REDUCTION", ignore_index=True)
_ypos = np.arange(len(_m))
fig, axes = plt.subplots(1, 2, figsize=(11, 0.55 * len(_m) + 2.2))
ax = axes[0]
ax.barh(_ypos + 0.19, _m["BASELINE_SPREAD"], height=0.36, color=AXIS_C,
        label="baseline (HS10 alone)")
ax.barh(_ypos - 0.19, _m["GROUPED_SPREAD"], height=0.36, color=PALETTE[0],
        label="grouped (HS10 + description group)")
ax.axvline(TARGET_SPREAD, color=INK2, linewidth=1.0, linestyle=":")
ax.annotate(f"target {TARGET_SPREAD}", xy=(TARGET_SPREAD, -0.55),
            xytext=(3, 0), textcoords="offset points", fontsize=8, color=INK2,
            va="bottom", annotation_clip=False)
ax.set_yticks(_ypos, _m["HS_CODE"])
ax.set_xlabel("robust spread of ln(PRICE_KG)")
ax.set_title("Price dispersion: baseline vs description groups")
ax.legend(loc="lower right", fontsize=8)

ax = axes[1]
_colors = [STATUS_COLORS.get(s, MUTED) for s in _m["STATUS"]]
ax.barh(_ypos, _m["DISPERSION_REDUCTION"].fillna(0.0), height=0.6, color=_colors)
for y, (red, st) in enumerate(zip(_m["DISPERSION_REDUCTION"], _m["STATUS"])):
    ax.annotate(f" {st} ({red:.0%})" if np.isfinite(red) else f" {st}",
                xy=(max(red, 0) if np.isfinite(red) else 0, y),
                xytext=(3, 0), textcoords="offset points",
                va="center", fontsize=8, color=INK)
ax.set_yticks(_ypos, _m["HS_CODE"])
ax.set_xlim(0, max(1.0, float(_m["DISPERSION_REDUCTION"].max() or 0) + 0.45))
ax.set_xlabel("dispersion reduction (1 − grouped/baseline)")
ax.set_title("Improvement and status per HS10")
plt.show()

expectation_df = master[["HS_CODE", "STATUS", "DISPERSION_REDUCTION",
                         "GROUPED_USABLE_COVERAGE"]].copy()
expectation_df["EXPECTED"] = expectation_df["HS_CODE"].map(
    EXPECTED_PROFILE).fillna("(not pre-classified)")
display(expectation_df.round(3))
print(master["STATUS"].value_counts().rename("HS10 count").to_frame())

In [ ]:
# Export the group assignments and the evaluation table.
EXPORT_GROUP_COLS = ["HS_CODE", "GROUP_ID", "point_id", "DECL_ID", "CMDT_ID",
                     "ACCURATE_NAME", "PRICE_KG", "MANUFACTURE_COUNTRY_CODE",
                     "TRANSPORT", "GROUP_SIZE"]
EXPORT_EVAL_COLS = ["HS_CODE", "N", "BASELINE_SPREAD", "GROUPED_SPREAD",
                    "DISPERSION_REDUCTION", "ETA_SQUARED",
                    "BASELINE_USABLE_COVERAGE", "GROUPED_USABLE_COVERAGE",
                    "CHOSEN_K", "MEDIAN_GROUP_SIZE", "MIN_GROUP_SIZE",
                    "SMALL_GROUP_SHARE", "STATUS", "METHOD", "APPROX_ARI"]

groups_export = pd.concat(all_groups, ignore_index=True)[EXPORT_GROUP_COLS]
groups_path = OUTPUT_PATH / "commodity_groups_embedding_only.parquet"
groups_export.to_parquet(groups_path, index=False)

eval_export = master[EXPORT_EVAL_COLS]
eval_path = OUTPUT_PATH / "hs10_grouping_evaluation.csv"
eval_export.to_csv(eval_path, index=False)

print(f"Wrote {groups_path}  ({groups_export.shape[0]:,} rows x "
      f"{groups_export.shape[1]} cols)")
print(f"Wrote {eval_path}  ({eval_export.shape[0]} rows x "
      f"{eval_export.shape[1]} cols)")
display(groups_export.head(10))

In [ ]:
# Final summary and per-HS10 interpretation.
_sum = master[["HS_CODE", "BASELINE_SPREAD", "GROUPED_SPREAD",
               "DISPERSION_REDUCTION", "ETA_SQUARED", "CHOSEN_K",
               "GROUPED_USABLE_COVERAGE", "STATUS"]].rename(columns={
    "BASELINE_SPREAD": "Baseline spread", "GROUPED_SPREAD": "Grouped spread",
    "DISPERSION_REDUCTION": "Reduction", "ETA_SQUARED": "eta2",
    "CHOSEN_K": "K", "GROUPED_USABLE_COVERAGE": "Coverage", "STATUS": "Status"})
display(_sum.round(3))

for hs in [h for h in batch_order if h in batch_diag]:
    diag = batch_diag[hs]
    m = diag["metrics"]
    sel = diag["sel"]
    sil = diag["sweep"].loc[diag["sweep"]["k"] == sel["chosen_k"],
                            "silhouette"].iloc[0]
    top_gid = diag["profile"]["GROUP_ID"].iloc[0]
    example = representative_names(diag["frame"], data[hs]["X"], top_gid, n=1)
    example_name = (example["ACCURATE_NAME"].iloc[0][:70]
                    if len(example) else "n/a")
    approx_note = ""
    if diag["method"] == "micro-approx":
        ari = (float(diag["approx"]["ARI"].min())
               if diag["approx"] is not None and len(diag["approx"]) else np.nan)
        approx_note = (f" Result comes from the micro-cluster APPROXIMATION "
                       f"(sample check ARI {ari:.2f} vs the exact method)."
                       if np.isfinite(ari) else
                       " Result comes from the micro-cluster APPROXIMATION.")
    print(f"\nHS10 {hs} -> {diag['status']}")
    print(f"  {diag['reason'].capitalize()}. Spread {m['baseline_spread']:.3f} -> "
          f"{m['grouped_spread']:.3f} (reduction {m['reduction']:.1%}) with k="
          f"{sel['chosen_k']}, coverage {m['coverage']:.1%}, median group size "
          f"{m['median_group_size']:.0f}, tiny-group share "
          f"{m['small_group_share']:.1%}.")
    print(f"  Embedding-side diagnostics: silhouette {sil:.3f} (diagnostic only); "
          f"largest group example: \"{example_name}\".{approx_note}")

print("\n" + "=" * 100)
print("Reading guide: a status label is a summary, not a verdict. Combine (1) the "
      "dispersion reduction, (2) group sizes and fragmentation, (3) coverage, "
      "(4) the representative descriptions (do the groups read as one commodity?), "
      "and (5) the embedding-side diagnostics before deciding whether description "
      "grouping should join HS10 + origin + transport for a given code.")